In [1]:
import collections
import csv
import time
from pathlib import Path

import numpy as np


BASE_PRETERMS = [chr(ord('A') + i) for i in range(26)]
BASE_PRETERMS[ord('s') - ord('a')] = 'PT_S'
WORD_TO_SYMBOL = {chr(ord('a') + i): BASE_PRETERMS[i] for i in range(26)}
SYMBOL_TO_WORD = {symbol: word for word, symbol in WORD_TO_SYMBOL.items()}


def load_symbol_corpus(path):
    corpus = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            toks = line.strip().split()
            if toks:
                corpus.append([WORD_TO_SYMBOL[tok] for tok in toks])
    return corpus


def count_adjacent_pairs(corpus):
    counts = collections.Counter()
    for sent in corpus:
        for i in range(len(sent) - 1):
            counts[(sent[i], sent[i + 1])] += 1
    return counts


def replace_pair_once(corpus, pair, new_symbol):
    updated = []
    replacements = 0
    left, right = pair
    for sent in corpus:
        out = []
        i = 0
        while i < len(sent):
            if i < len(sent) - 1 and sent[i] == left and sent[i + 1] == right:
                out.append(new_symbol)
                replacements += 1
                i += 2
            else:
                out.append(sent[i])
                i += 1
        updated.append(out)
    return updated, replacements


def apply_pair_until_stable(corpus, pair, new_symbol, max_passes=1000):
    current = corpus
    passes = 0
    total_replacements = 0
    for _ in range(max_passes):
        current, replacements = replace_pair_once(current, pair, new_symbol)
        passes += 1
        total_replacements += replacements
        if replacements == 0:
            break
    return current, passes, total_replacements


def segment_to_pair_symbols(sentence, pair_to_symbol):
    segmented = []
    i = 0
    while i < len(sentence):
        if i + 1 < len(sentence) and (sentence[i], sentence[i + 1]) in pair_to_symbol:
            segmented.append(pair_to_symbol[(sentence[i], sentence[i + 1])])
            i += 2
        else:
            return None, i
    return segmented, None


def corpus_stats(corpus):
    lengths = np.array([len(sent) for sent in corpus], dtype=np.int32)
    return {
        'num_sentences': len(corpus),
        'min_len': int(lengths.min()),
        'max_len': int(lengths.max()),
        'mean_len': float(lengths.mean()),
        'median_len': float(np.median(lengths)),
    }


def write_symbol_corpus(corpus, path):
    with open(path, 'w', encoding='utf-8') as f:
        for sent in corpus:
            f.write(' '.join(sent) + '\n')


print('Utilities loaded.')
print(f'  Base preterminals: {BASE_PRETERMS}')


Utilities loaded.
  Base preterminals: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'PT_S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z']


In [ ]:
CORPUS_PATH = 'sample/pcfg3_10k.txt'
PAIR_CORPUS_PATH = 'sample/pcfg3_10k_lexicalcollapsed.txt'
FAMILY_CORPUS_PATH = 'sample/pcfg3_10k_familycollapsed.txt'

LEXICAL_PAIR_SYMBOLS = collections.OrderedDict([
    (('A', 'B'), 'LP_AB'),
    (('A', 'M'), 'LP_AM'),
    (('C', 'D'), 'LP_CD'),
    (('E', 'F'), 'LP_EF'),
    (('E', 'I'), 'LP_EI'),
    (('G', 'H'), 'LP_GH'),
    (('I', 'J'), 'LP_IJ'),
    (('K', 'L'), 'LP_KL'),
    (('M', 'N'), 'LP_MN'),
    (('O', 'P'), 'LP_OP'),
    (('O', 'U'), 'LP_OU'),
    (('PT_S', 'T'), 'LP_ST'),
    (('Q', 'R'), 'LP_QR'),
    (('U', 'V'), 'LP_UV'),
    (('W', 'X'), 'LP_WX'),
    (('Y', 'Z'), 'LP_YZ'),
])
PAIR_SYMBOL_TO_RHS = {symbol: pair for pair, symbol in LEXICAL_PAIR_SYMBOLS.items()}


FAMILY_GROUPS = collections.OrderedDict([
    ('F_ABCD', ['LP_AB', 'LP_CD']),
    ('F_EFGH', ['LP_EF', 'LP_GH']),
    ('F_EIOU', ['LP_EI', 'LP_OU']),
    ('F_IJKL', ['LP_IJ', 'LP_KL']),
    ('F_MNOP', ['LP_MN', 'LP_OP']),
    ('F_QRST', ['LP_QR', 'LP_ST']),
    ('F_UVWX', ['LP_UV', 'LP_WX']),
    ('F_AM', ['LP_AM']),
    ('F_YZ', ['LP_YZ']),
])

PAIR_TO_FAMILY = {}
for family_symbol, pair_symbols in FAMILY_GROUPS.items():
    for pair_symbol in pair_symbols:
        if pair_symbol in PAIR_TO_FAMILY:
            raise ValueError(f'duplicate family assignment for {pair_symbol}')
        PAIR_TO_FAMILY[pair_symbol] = family_symbol

missing_pairs = [symbol for symbol in PAIR_SYMBOL_TO_RHS if symbol not in PAIR_TO_FAMILY]
if missing_pairs:
    raise ValueError(f'unassigned lexical pairs: {missing_pairs}')

raw_corpus = load_symbol_corpus(CORPUS_PATH)
raw_pair_counts = count_adjacent_pairs(raw_corpus)
raw_stats = corpus_stats(raw_corpus)

ordered_pairs = sorted(
    LEXICAL_PAIR_SYMBOLS.items(),
    key=lambda item: (-raw_pair_counts[item[0]], item[1]),
)

print('Pair inventory and raw frequencies:')
for pair, pair_symbol in ordered_pairs:
    print(f"  {pair_symbol:>6s} -> {pair[0]:>5s} {pair[1]:<5s}  raw_count={raw_pair_counts[pair]:5d}")

print('\nFamily groups:')
for family_symbol, pair_symbols in FAMILY_GROUPS.items():
    print(f"  {family_symbol:<8s} <- {', '.join(pair_symbols)}")

pair_corpus = []
family_corpus = []
failed_segmentations = []
pair_symbol_counts = collections.Counter()
family_symbol_counts = collections.Counter()
family_pair_choice_counts = {
    family_symbol: collections.Counter()
    for family_symbol in FAMILY_GROUPS
}

for sent_idx, sent in enumerate(raw_corpus):
    segmented_pairs, fail_pos = segment_to_pair_symbols(sent, LEXICAL_PAIR_SYMBOLS)
    if segmented_pairs is None:
        failed_segmentations.append({
            'sentence_index': sent_idx,
            'fail_pos': fail_pos,
            'sentence': sent,
        })
        continue

    pair_corpus.append(segmented_pairs)
    pair_symbol_counts.update(segmented_pairs)

    family_sent = [PAIR_TO_FAMILY[pair_symbol] for pair_symbol in segmented_pairs]
    family_corpus.append(family_sent)
    family_symbol_counts.update(family_sent)

    for pair_symbol, family_symbol in zip(segmented_pairs, family_sent):
        family_pair_choice_counts[family_symbol][pair_symbol] += 1

pair_stats = corpus_stats(pair_corpus)
family_stats = corpus_stats(family_corpus)
covered_sentences = len(pair_corpus)
dropped_sentences = len(failed_segmentations)
coverage_pct = 100.0 * covered_sentences / len(raw_corpus)

write_symbol_corpus(pair_corpus, PAIR_CORPUS_PATH)
write_symbol_corpus(family_corpus, FAMILY_CORPUS_PATH)

FAMILY_TO_PAIR_PROBS = collections.OrderedDict()
for family_symbol, pair_symbols in FAMILY_GROUPS.items():
    counts = family_pair_choice_counts[family_symbol]
    total = sum(counts.values())
    if total == 0:
        continue
    FAMILY_TO_PAIR_PROBS[family_symbol] = collections.OrderedDict(
        (pair_symbol, counts[pair_symbol] / total)
        for pair_symbol in pair_symbols
        if counts[pair_symbol] > 0
    )

print('\nRaw corpus stats:', raw_stats)
print('Pair-only corpus stats:', pair_stats)
print('Family-only corpus stats:', family_stats)
print(f'Covered sentences: {covered_sentences} / {len(raw_corpus)} ({coverage_pct:.2f}%)')
print(f'Dropped sentences: {dropped_sentences}')
print(f'Pair-only corpus written to   {PAIR_CORPUS_PATH}')
print(f'Family-only corpus written to {FAMILY_CORPUS_PATH}')

if failed_segmentations:
    print('\nFirst 5 uncovered sentences:')
    for item in failed_segmentations[:5]:
        fail_pos = item['fail_pos']
        sent = item['sentence']
        context = ' '.join(sent[max(0, fail_pos - 2):fail_pos + 3])
        print(
            f"  idx={item['sentence_index']:5d}  fail_pos={fail_pos:2d}  "
            f"token={sent[fail_pos]!r}  context={context}"
        )


Pair inventory and raw frequencies:
   LP_QR ->     Q R      raw_count=21748
   LP_YZ ->     Y Z      raw_count=20435
   LP_UV ->     U V      raw_count=14879
   LP_EI ->     E I      raw_count=14820
   LP_AB ->     A B      raw_count=14772
   LP_EF ->     E F      raw_count= 9256
   LP_ST ->  PT_S T      raw_count= 9195
   LP_AM ->     A M      raw_count= 8556
   LP_IJ ->     I J      raw_count= 7354
   LP_OU ->     O U      raw_count= 6368
   LP_WX ->     W X      raw_count= 6363
   LP_CD ->     C D      raw_count= 6164
   LP_MN ->     M N      raw_count= 4814
   LP_GH ->     G H      raw_count= 3944
   LP_KL ->     K L      raw_count= 2857
   LP_OP ->     O P      raw_count= 1876

Hint-driven family groups:
  F_ABCD   <- LP_AB, LP_CD
  F_EFGH   <- LP_EF, LP_GH
  F_EIOU   <- LP_EI, LP_OU
  F_IJKL   <- LP_IJ, LP_KL
  F_MNOP   <- LP_MN, LP_OP
  F_QRST   <- LP_QR, LP_ST
  F_UVWX   <- LP_UV, LP_WX
  F_AM     <- LP_AM
  F_YZ     <- LP_YZ

Raw corpus stats: {'num_sentences': 10000, 'min_le

In [ ]:
present_pair_symbols = [symbol for symbol in LEXICAL_PAIR_SYMBOLS.values() if pair_symbol_counts[symbol] > 0]
present_families = [family_symbol for family_symbol in FAMILY_GROUPS if family_symbol in family_symbol_counts]
family_adjacent_counts = count_adjacent_pairs(family_corpus)

print(f'Observed pair symbols after full segmentation ({len(present_pair_symbols)}):')
print(present_pair_symbols)

print(f'\nObserved family symbols after grouping ({len(present_families)}):')
print(present_families)

print('\nLength profile:')
print(f"  raw mean/median/max:    {raw_stats['mean_len']:.2f} / {raw_stats['median_len']:.0f} / {raw_stats['max_len']}")
print(f"  pair mean/median/max:   {pair_stats['mean_len']:.2f} / {pair_stats['median_len']:.0f} / {pair_stats['max_len']}")
print(f"  family mean/median/max: {family_stats['mean_len']:.2f} / {family_stats['median_len']:.0f} / {family_stats['max_len']}")

raw_obs = len(BASE_PRETERMS)
pair_obs = len(present_pair_symbols)
family_obs = len(present_families)
print('\nDense observed child-pair search space:')
print(f'  raw observed inventory:    {raw_obs} symbols -> {raw_obs * raw_obs:,} child pairs')
print(f'  lexical-pair inventory:    {pair_obs} symbols -> {pair_obs * pair_obs:,} child pairs')
print(f'  family-symbol inventory:   {family_obs} symbols -> {family_obs * family_obs:,} child pairs')

print('\nFamily-to-base-pair mixtures:')
for family_symbol in present_families:
    rhs_probs = FAMILY_TO_PAIR_PROBS[family_symbol]
    rhs_str = ', '.join(f'{pair_symbol} ({prob:.3f})' for pair_symbol, prob in rhs_probs.items())
    print(f'  {family_symbol:<8s} -> {rhs_str}')

print('\nTop 30 adjacent family pairs after grouping:')
for (left, right), count in family_adjacent_counts.most_common(30):
    print(f'  {left:<8s} {right:<8s}  count={count}')

print('\nFirst pair-only sentence:')
print(' '.join(pair_corpus[0]))
print('\nFirst family-only sentence:')
print(' '.join(family_corpus[0]))


Observed pair symbols after full segmentation (16):
['LP_AB', 'LP_AM', 'LP_CD', 'LP_EF', 'LP_EI', 'LP_GH', 'LP_IJ', 'LP_KL', 'LP_MN', 'LP_OP', 'LP_OU', 'LP_ST', 'LP_QR', 'LP_UV', 'LP_WX', 'LP_YZ']

Observed family symbols after hint-driven grouping (9):
['F_ABCD', 'F_EFGH', 'F_EIOU', 'F_IJKL', 'F_MNOP', 'F_QRST', 'F_UVWX', 'F_AM', 'F_YZ']

Length profile:
  raw mean/median/max:    30.68 / 32 / 80
  pair mean/median/max:   15.33 / 16 / 40
  family mean/median/max: 15.33 / 16 / 40

Dense observed child-pair search space:
  raw observed inventory:    26 symbols -> 676 child pairs
  lexical-pair inventory:    16 symbols -> 256 child pairs
  family-symbol inventory:   9 symbols -> 81 child pairs

Family-to-base-pair mixtures:
  F_ABCD   -> LP_AB (0.706), LP_CD (0.294)
  F_EFGH   -> LP_EF (0.701), LP_GH (0.299)
  F_EIOU   -> LP_EI (0.699), LP_OU (0.301)
  F_IJKL   -> LP_IJ (0.720), LP_KL (0.280)
  F_MNOP   -> LP_MN (0.719), LP_OP (0.281)
  F_QRST   -> LP_QR (0.703), LP_ST (0.297)
  F_UVWX   

In [4]:
class GeneralPCFG_EM:
    def __init__(self, observed_symbols, n_nt):
        self.observed_symbols = list(observed_symbols)
        self.obs_to_idx = {sym: i for i, sym in enumerate(self.observed_symbols)}
        self.n_obs = len(self.observed_symbols)
        self.n_nt = n_nt
        self.N = self.n_obs + n_nt
        self.S = self.n_obs
        self.rules = np.zeros((n_nt, self.N, self.N), dtype=np.float64)

    def sym_name(self, idx):
        if idx < self.n_obs:
            return self.observed_symbols[idx]
        if idx == self.S:
            return 'S'
        return f"NT{idx - self.n_obs}"

    def encode_corpus(self, symbol_corpus):
        encoded = []
        for sent in symbol_corpus:
            encoded.append(np.array([self.obs_to_idx[sym] for sym in sent], dtype=np.int32))
        return encoded

    def init_random(self, seed=42):
        rng = np.random.default_rng(seed)
        self.rules[:] = rng.random(self.rules.shape)
        self._normalise()

    def clone(self):
        copied = GeneralPCFG_EM(self.observed_symbols, self.n_nt)
        copied.rules = self.rules.copy()
        return copied

    def _normalise(self):
        for a in range(self.n_nt):
            total = self.rules[a].sum()
            if total > 0:
                self.rules[a] /= total

    def inside(self, sent):
        n = len(sent)
        alpha = np.zeros((self.N, n, n), dtype=np.float64)
        for i in range(n):
            alpha[sent[i], i, i] = 1.0

        for width in range(2, n + 1):
            for i in range(n - width + 1):
                j = i + width - 1
                for k in range(i, j):
                    left = alpha[:, i, k]
                    right = alpha[:, k + 1, j]
                    outer = np.outer(left, right)
                    alpha[self.n_obs:, i, j] += np.einsum('abc,bc->a', self.rules, outer)
        return alpha

    def outside(self, sent, alpha):
        n = len(sent)
        beta = np.zeros((self.N, n, n), dtype=np.float64)
        beta[self.S, 0, n - 1] = 1.0

        for width in range(n, 1, -1):
            for i in range(n - width + 1):
                j = i + width - 1
                parent_beta = beta[self.n_obs:, i, j]
                if not np.any(parent_beta):
                    continue
                for k in range(i, j):
                    left_alpha = alpha[:, i, k]
                    right_alpha = alpha[:, k + 1, j]
                    wr = np.einsum('alr,r->al', self.rules, right_alpha)
                    beta[:, i, k] += parent_beta @ wr
                    wl = np.einsum('alr,l->ar', self.rules, left_alpha)
                    beta[:, k + 1, j] += parent_beta @ wl
        return beta

    def em_step(self, sentences, max_len=30):
        counts = np.zeros_like(self.rules)
        ll = 0.0
        used = 0

        for sent in sentences:
            n = len(sent)
            if n < 2 or n > max_len:
                continue

            alpha = self.inside(sent)
            sentence_prob = alpha[self.S, 0, n - 1]
            if sentence_prob < 1e-300:
                continue

            ll += np.log(sentence_prob)
            used += 1
            beta = self.outside(sent, alpha)

            for width in range(2, n + 1):
                for i in range(n - width + 1):
                    j = i + width - 1
                    parent_beta = beta[self.n_obs:, i, j]
                    if not np.any(parent_beta):
                        continue
                    for k in range(i, j):
                        left = alpha[:, i, k]
                        right = alpha[:, k + 1, j]
                        outer = np.outer(left, right)
                        counts += (parent_beta[:, None, None] * self.rules * outer[None, :, :]) / sentence_prob

        for a in range(self.n_nt):
            total = counts[a].sum()
            if total > 0:
                self.rules[a] = counts[a] / total

        return ll, used

    def prune(self, threshold=0.02):
        self.rules[self.rules < threshold] = 0.0
        self._normalise()

    def count_rules(self, threshold=1e-10):
        return int(np.sum(self.rules > threshold))


def run_schedule(model, encoded_corpus, phases, seed=42):
    rng = np.random.default_rng(seed)
    sorted_corpus = sorted(encoded_corpus, key=len)
    history = []
    global_iter = 0

    for phase_idx, phase in enumerate(phases, start=1):
        name = phase['name']
        max_len = phase['max_len']
        iters = phase['iters']
        batch = phase.get('batch', 0)
        prune_every = phase.get('prune_every', 0)
        soft_prune = phase.get('soft_prune', 0.0)
        eligible = [sent for sent in sorted_corpus if 2 <= len(sent) <= max_len]

        print(f"\nPhase {phase_idx}: {name}")
        print(f"  max_len={max_len}  iters={iters}  batch={batch or 'all'}  eligible={len(eligible)}")
        if prune_every and soft_prune > 0:
            print(f"  early prune every {prune_every} iters at threshold {soft_prune:.4f}")
        else:
            print('  no early pruning in this phase')

        for phase_iter in range(1, iters + 1):
            global_iter += 1
            if batch and batch < len(eligible):
                idxs = rng.choice(len(eligible), size=batch, replace=False)
                batch_sents = [eligible[i] for i in idxs]
            else:
                batch_sents = eligible

            t0 = time.time()
            ll, used = model.em_step(batch_sents, max_len=max_len)
            dt = time.time() - t0
            avg_ll = ll / max(used, 1)
            nr = model.count_rules()
            history.append({
                'iter': global_iter,
                'phase': name,
                'phase_iter': phase_iter,
                'avg_ll': avg_ll,
                'rules': nr,
                'used': used,
                'time': dt,
            })
            print(
                f"  [{global_iter:3d}] phase_iter={phase_iter:2d}/{iters}  "
                f"avg_LL={avg_ll:8.3f}  sents={used:4d}  rules={nr:5d}  {dt:5.1f}s"
            )

            if prune_every and soft_prune > 0 and phase_iter % prune_every == 0:
                before = model.count_rules()
                model.prune(threshold=soft_prune)
                after = model.count_rules()
                print(f"         -> early prune: {before} -> {after} rules")

    return history


def export_expanded_csv(model, output_path, family_to_pair_probs, pair_symbol_to_rhs, threshold=1e-10):
    rows = []
    rule_id = 1

    for base_symbol in BASE_PRETERMS:
        rows.append({
            'ID': rule_id,
            'LHS': base_symbol,
            'LHS Type': 'preterminal',
            'RHS': SYMBOL_TO_WORD[base_symbol],
            'Probability': 1.0,
        })
        rule_id += 1

    for family_symbol, rhs_probs in family_to_pair_probs.items():
        if family_symbol not in model.obs_to_idx:
            continue
        for pair_symbol, prob in rhs_probs.items():
            left, right = pair_symbol_to_rhs[pair_symbol]
            rows.append({
                'ID': rule_id,
                'LHS': family_symbol,
                'LHS Type': 'nonterminal',
                'RHS': f'{left} {right}',
                'Probability': round(float(prob), 6),
            })
            rule_id += 1

    for a in range(model.n_nt):
        for b in range(model.N):
            for c in range(model.N):
                prob = model.rules[a, b, c]
                if prob > threshold:
                    rows.append({
                        'ID': rule_id,
                        'LHS': model.sym_name(a + model.n_obs),
                        'LHS Type': 'nonterminal',
                        'RHS': f"{model.sym_name(b)} {model.sym_name(c)}",
                        'Probability': round(float(prob), 6),
                    })
                    rule_id += 1

    with open(output_path, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=['ID', 'LHS', 'LHS Type', 'RHS', 'Probability'])
        writer.writeheader()
        writer.writerows(rows)

    return rows


def export_seed_pcfg(model, output_path, family_to_pair_probs, pair_symbol_to_rhs, threshold=1e-10):
    lines = []
    for family_symbol, rhs_probs in family_to_pair_probs.items():
        if family_symbol not in model.obs_to_idx:
            continue
        for pair_symbol, prob in rhs_probs.items():
            left, right = pair_symbol_to_rhs[pair_symbol]
            lines.append(f'{family_symbol} -> {left} {right} [{prob:.6f}]')
    for a in range(model.n_nt):
        lhs = model.sym_name(a + model.n_obs)
        for b in range(model.N):
            for c in range(model.N):
                prob = model.rules[a, b, c]
                if prob > threshold:
                    rhs = f'{model.sym_name(b)} {model.sym_name(c)}'
                    lines.append(f'{lhs} -> {rhs} [{prob:.6f}]')
    with open(output_path, 'w', encoding='utf-8') as f:
        f.write('\n'.join(lines) + '\n')


def build_family_level_pcfg(model, family_to_pair_probs, pair_symbol_to_rhs, threshold=1e-10):
    from nltk.grammar import Nonterminal, PCFG, ProbabilisticProduction

    rules = []
    for base_symbol in BASE_PRETERMS:
        lhs = Nonterminal(base_symbol)
        rhs = [SYMBOL_TO_WORD[base_symbol]]
        rules.append(ProbabilisticProduction(lhs, rhs, prob=1.0))

    for family_symbol, rhs_probs in family_to_pair_probs.items():
        if family_symbol not in model.obs_to_idx:
            continue
        lhs = Nonterminal(family_symbol)
        for pair_symbol, prob in rhs_probs.items():
            left, right = pair_symbol_to_rhs[pair_symbol]
            rhs = [Nonterminal(left), Nonterminal(right)]
            rules.append(ProbabilisticProduction(lhs, rhs, prob=float(prob)))

    for a in range(model.n_nt):
        lhs = Nonterminal(model.sym_name(a + model.n_obs))
        for b in range(model.N):
            for c in range(model.N):
                prob = model.rules[a, b, c]
                if prob > threshold:
                    rhs = [Nonterminal(model.sym_name(b)), Nonterminal(model.sym_name(c))]
                    rules.append(ProbabilisticProduction(lhs, rhs, prob=float(prob)))

    return PCFG(Nonterminal('S'), rules)


print('Generalized EM utilities loaded.')


Generalized EM utilities loaded.


In [ ]:
# EM now observes family symbols, not individual LP_* symbols.
observed_symbols = [family_symbol for family_symbol in FAMILY_GROUPS if family_symbol in present_families]

N_NT = 9
SEED = 21
PHASES = [
    {'name': 'short_warmup', 'max_len': 10, 'iters': 8, 'batch': 2000, 'prune_every': 2, 'soft_prune': 0.0005},
    {'name': 'medium_expand', 'max_len': 16, 'iters': 8, 'batch': 2500, 'prune_every': 4, 'soft_prune': 0.0010},
    {'name': 'long_refine', 'max_len': 24, 'iters': 10, 'batch': 3000, 'prune_every': 5, 'soft_prune': 0.0015},
]

model = GeneralPCFG_EM(observed_symbols, N_NT)
model.init_random(seed=SEED)
encoded_corpus = model.encode_corpus(family_corpus)

print(f'Observed family inventory for EM: {len(observed_symbols)}')
print(observed_symbols)
print(f'Rule tensor size: {model.n_nt} x {model.N} x {model.N} = {model.n_nt * model.N * model.N:,}')

history = run_schedule(model, encoded_corpus, PHASES, seed=SEED)
print(f"\nTraining complete. Active rules: {model.count_rules()}")


Observed family inventory for EM: 9
['F_ABCD', 'F_EFGH', 'F_EIOU', 'F_IJKL', 'F_MNOP', 'F_QRST', 'F_UVWX', 'F_AM', 'F_YZ']
Rule tensor size: 13 x 22 x 22 = 6,292

Phase 1: short_warmup
  max_len=10  iters=8  batch=2000  eligible=3204
  early prune every 2 iters at threshold 0.0005
  [  1] phase_iter= 1/8  avg_LL= -18.946  sents=2000  rules= 6123    5.9s
  [  2] phase_iter= 2/8  avg_LL= -14.628  sents=2000  rules= 6110    6.1s
         -> early prune: 6110 -> 3882 rules
  [  3] phase_iter= 3/8  avg_LL= -13.330  sents=2000  rules= 3882    8.8s
  [  4] phase_iter= 4/8  avg_LL= -12.108  sents=2000  rules= 3882   35.0s
         -> early prune: 3882 -> 2248 rules
  [  5] phase_iter= 5/8  avg_LL= -10.835  sents=2000  rules= 2248   32.3s
  [  6] phase_iter= 6/8  avg_LL= -10.138  sents=2000  rules= 2246   28.2s
         -> early prune: 2246 -> 1182 rules
  [  7] phase_iter= 7/8  avg_LL=  -9.742  sents=1999  rules= 1182    3.3s
  [  8] phase_iter= 8/8  avg_LL=  -9.470  sents=1999  rules= 1181   

In [6]:
OUTPUT_CSV = 'pcfg3.csv'
OUTPUT_PCFG = 'task3_seed.pcfg'
FINAL_PRUNE_THRESHOLD = 0.02
SHOW_TOP_N = 60

export_model = model.clone()
before = export_model.count_rules()
export_model.prune(threshold=FINAL_PRUNE_THRESHOLD)
after = export_model.count_rules()

print(f'Final prune: {before} -> {after} rules at threshold {FINAL_PRUNE_THRESHOLD:.3f}')
rows = export_expanded_csv(export_model, OUTPUT_CSV, FAMILY_TO_PAIR_PROBS, PAIR_SYMBOL_TO_RHS)
export_seed_pcfg(export_model, OUTPUT_PCFG, FAMILY_TO_PAIR_PROBS, PAIR_SYMBOL_TO_RHS)

family_rows = [
    row for row in rows
    if row['LHS Type'] == 'nonterminal' and row['LHS'] in observed_symbols
]
family_rows = sorted(family_rows, key=lambda row: (row['LHS'], -row['Probability'], row['RHS']))

latent_rows = [
    row for row in rows
    if row['LHS Type'] == 'nonterminal' and row['LHS'] not in observed_symbols
]
latent_rows = sorted(latent_rows, key=lambda row: row['Probability'], reverse=True)

print(f'Expanded grammar written to {OUTPUT_CSV}')
print(f'Seed grammar string written to {OUTPUT_PCFG}')
print(f"Total rows: {len(rows)}  Nonterminal rows: {len(family_rows) + len(latent_rows)}")

print('\nFamily expansions:')
for row in family_rows:
    print(f"  {row['LHS']:>8s} -> {row['RHS']:<18s}  P={row['Probability']:.4f}")

print(f'\nTop {min(SHOW_TOP_N, len(latent_rows))} latent rules:')
for row in latent_rows[:SHOW_TOP_N]:
    print(f"  {row['LHS']:>8s} -> {row['RHS']:<18s}  P={row['Probability']:.4f}")


Final prune: 99 -> 59 rules at threshold 0.020
Expanded grammar written to pcfg3.csv
Seed grammar string written to task3_seed.pcfg
Total rows: 101  Nonterminal rows: 75

Family expansions:
    F_ABCD -> A B                 P=0.7057
    F_ABCD -> C D                 P=0.2943
      F_AM -> A M                 P=1.0000
    F_EFGH -> E F                 P=0.7013
    F_EFGH -> G H                 P=0.2987
    F_EIOU -> E I                 P=0.6994
    F_EIOU -> O U                 P=0.3006
    F_IJKL -> I J                 P=0.7203
    F_IJKL -> K L                 P=0.2797
    F_MNOP -> M N                 P=0.7194
    F_MNOP -> O P                 P=0.2806
    F_QRST -> Q R                 P=0.7028
    F_QRST -> PT_S T              P=0.2972
    F_UVWX -> U V                 P=0.7006
    F_UVWX -> W X                 P=0.2994
      F_YZ -> Y Z                 P=1.0000

Top 59 latent rules:
       NT9 -> F_EIOU F_UVWX       P=0.8462
       NT7 -> NT8 NT3             P=0.7240
      NT11 -> 

In [7]:
# Evaluate exact pair-resolved strings under the exported family grammar.
# The higher-level inside score is computed on family symbols, then we add the
# log-probability of each family choosing the observed base pair.

EVAL_SENTENCES = min(10000, len(family_corpus))  # set to len(family_corpus) for a full pass
eval_family_corpus = family_corpus[:EVAL_SENTENCES]
eval_pair_corpus = pair_corpus[:EVAL_SENTENCES]
encoded_eval = export_model.encode_corpus(eval_family_corpus)

parsed = 0
failed = 0
total_log2_prob = 0.0
fail_by_len = collections.Counter()

for idx, (family_sent, pair_sent, encoded_sent) in enumerate(zip(eval_family_corpus, eval_pair_corpus, encoded_eval), start=1):
    alpha = export_model.inside(encoded_sent)
    family_prob = alpha[export_model.S, 0, len(encoded_sent) - 1]

    if family_prob > 1e-300:
        lexical_log2 = 0.0
        lexical_ok = True
        for family_symbol, pair_symbol in zip(family_sent, pair_sent):
            pair_prob = FAMILY_TO_PAIR_PROBS[family_symbol].get(pair_symbol, 0.0)
            if pair_prob <= 1e-300:
                lexical_ok = False
                break
            lexical_log2 += np.log2(pair_prob)

        if lexical_ok:
            total_log2_prob += np.log2(family_prob) + lexical_log2
            parsed += 1
        else:
            failed += 1
            fail_by_len[len(pair_sent)] += 1
    else:
        failed += 1
        fail_by_len[len(pair_sent)] += 1

    if idx % 200 == 0:
        print(f'Processed {idx}/{EVAL_SENTENCES} family-grouped sentences...')

coverage = 100.0 * parsed / max(parsed + failed, 1)
avg_log2_prob = total_log2_prob / max(parsed, 1)

print('\n' + '=' * 40)
print('FAMILY-LEVEL EVALUATION')
print('=' * 40)
print(f'Pair-segmented sentences available:  {len(pair_corpus)}')
print(f'Dropped before segmentation:        {dropped_sentences}')
print(f'Evaluated pair-resolved strings:    {EVAL_SENTENCES}')
print(f'Successfully scored:                {parsed} / {EVAL_SENTENCES}')
print(f'Failed:                             {failed}')
print(f'Coverage:                           {coverage:.2f}%')
print(f'Average Log2 Likelihood:            {avg_log2_prob:.4f}')

if failed > 0:
    print('\nFailures by pair length:')
    for length, count in fail_by_len.most_common(10):
        print(f'  len={length:2d}: {count} failures')


Processed 200/9987 family-grouped sentences...
Processed 400/9987 family-grouped sentences...
Processed 600/9987 family-grouped sentences...
Processed 800/9987 family-grouped sentences...
Processed 1000/9987 family-grouped sentences...
Processed 1200/9987 family-grouped sentences...
Processed 1400/9987 family-grouped sentences...
Processed 1600/9987 family-grouped sentences...
Processed 1800/9987 family-grouped sentences...
Processed 2000/9987 family-grouped sentences...
Processed 2200/9987 family-grouped sentences...
Processed 2400/9987 family-grouped sentences...
Processed 2600/9987 family-grouped sentences...
Processed 2800/9987 family-grouped sentences...
Processed 3000/9987 family-grouped sentences...
Processed 3200/9987 family-grouped sentences...
Processed 3400/9987 family-grouped sentences...
Processed 3600/9987 family-grouped sentences...
Processed 3800/9987 family-grouped sentences...
Processed 4000/9987 family-grouped sentences...
Processed 4200/9987 family-grouped sentences